## 03 — Weighted Aggregation

Produces five pre-aggregated summary CSVs (one per visualisation story) for use by Processing rendering sketches. All percentages are weighted using the CSBS `weight` variable. Missing/routing codes (`-1`, `-97`, `-99`) are excluded before any computation.

Two governance composites are included in each relevant output so the visualisation can switch between them:
- **Score A (binary sum):** each governance indicator treated as 0/1, summed and normalised 0–1. Priority dichotomised: 1 or 2 = high (1), 3 or 4 = low (0).
- **Score B (continuous):** three-component average (each 0–1). Priority normalised linearly: `(4 − priority) / 3`. Manage and policy components are proportions of positive items.

**Governance variables:**
- Priority: `priority` (Q9, ordinal 1=Very high → 4=Very low)
- Manage (positive indicators): `manage1`, `manage2`, `manage3`, `manage4`
- Manage (negative / exclusion): `manage6` (DK), `manage7` (None) — not scored
- Policy (positive indicators): `policy1`–`policy5` — routing code `-1` treated as 0 (no policy → no coverage)
- Policy (negative / exclusion): `policy8` (DK), `policy9` (None) — not scored

**Breach flag:** `type11 == 0` (type11 = 'None of these' at Q53A; equals 0 when any breach type occurred).

**Stable core used:** 172 variables present across all 8 waves. All variables here are in that core.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data_raw"
OUT_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\toby\OneDrive - Node9\Local Github Repo\CSBS Project


In [14]:
MISSING_CODES = (-1, -97, -99)

MANAGE_POSITIVE = ["manage1", "manage2", "manage3", "manage4"]
POLICY_POSITIVE = ["policy1", "policy2", "policy3", "policy4", "policy5"]

SIZE_LABELS = {1: "Micro", 2: "Small", 3: "Medium", 4: "Large"}

def normalise_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
    )
    return df

---
### 1. Load all waves

In [15]:
years = list(range(2018, 2026))
tab_files = {year: RAW_DIR / f"{year}.tab" for year in years}

missing = [y for y, p in tab_files.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing tab files: {missing}")

data = {}
for year, path in tab_files.items():
    df = pd.read_csv(path, sep="\t", low_memory=False)
    df = normalise_columns(df)
    df["year"] = year
    data[year] = df

print({y: df.shape for y, df in data.items()})

{2018: (2088, 462), 2019: (2080, 462), 2020: (1900, 313), 2021: (2284, 421), 2022: (2157, 447), 2023: (3991, 568), 2024: (3434, 499), 2025: (3835, 528)}


---
### 2. Per-respondent derived indicators

Added to each wave before stacking:
- `priority_bin`, `priority_norm` — binary and normalised priority
- `manage_sum`, `manage_prop` — count and proportion of positive manage items
- `policy_sum`, `policy_prop` — count and proportion of positive policy items (routing -1 → 0)
- `governance_A`, `governance_B` — the two composite scores
- `breached` — binary breach flag
- `has_policy` — manage3==1
- `actively_managed` — manage1==1 (board/senior manager with cyber responsibility)
- `org_size_label` — string label for sizeb

In [16]:
def add_indicators(df):
    df = df.copy()

    # --- Priority ---
    priority = pd.to_numeric(df["priority"], errors="coerce")
    priority = priority.where(~priority.isin(MISSING_CODES))
    df["priority_bin"] = np.where(priority.notna(), (priority <= 2).astype(float), np.nan)
    df["priority_norm"] = (4 - priority) / 3  # 1→1.0, 2→0.667, 3→0.333, 4→0.0; NaN propagates

    # --- Manage: routed out if manage1 in MISSING_CODES ---
    manage_routed = df["manage1"].isin(MISSING_CODES)
    manage_df = df[MANAGE_POSITIVE].apply(pd.to_numeric, errors="coerce")
    manage_df = manage_df.where(~manage_df.isin(MISSING_CODES))  # routing → NaN
    df["manage_sum"] = manage_df.sum(axis=1, skipna=True, min_count=1).where(~manage_routed)
    df["manage_prop"] = df["manage_sum"] / len(MANAGE_POSITIVE)

    # --- Policy: -1 routing → 0 (no policy = no coverage); -97/-99 → NaN ---
    policy_df = df[POLICY_POSITIVE].apply(pd.to_numeric, errors="coerce")
    true_missing_policy = policy_df.isin((-97, -99))
    policy_df = policy_df.where(policy_df != -1, other=0)   # routing -1 → 0
    policy_df = policy_df.where(~true_missing_policy)       # -97/-99 → NaN
    df["policy_sum"] = policy_df.sum(axis=1, skipna=True, min_count=1).where(~manage_routed)
    df["policy_prop"] = df["policy_sum"] / len(POLICY_POSITIVE)

    # --- Composite A: (priority_bin + manage_sum + policy_sum) / 10 ---
    df["governance_A"] = (df["priority_bin"] + df["manage_sum"] + df["policy_sum"]) / 10

    # --- Composite B: average of three normalised components (each 0–1) ---
    df["governance_B"] = (
        df["priority_norm"] + df["manage_prop"] + df["policy_prop"]
    ) / 3

    # --- Breach flag: type11==0 means at least one breach type occurred ---
    type11 = pd.to_numeric(df["type11"], errors="coerce")
    type11 = type11.where(~type11.isin(MISSING_CODES))
    df["breached"] = np.where(type11.notna(), (type11 == 0).astype(float), np.nan)

    # --- Pathway indicators (Story 3) ---
    m3 = pd.to_numeric(df["manage3"], errors="coerce")
    m1 = pd.to_numeric(df["manage1"], errors="coerce")
    df["has_policy"]       = np.where(m3.isin(MISSING_CODES), np.nan, (m3 == 1).astype(float))
    df["actively_managed"] = np.where(m1.isin(MISSING_CODES), np.nan, (m1 == 1).astype(float))

    # --- Org size label ---
    sizeb = pd.to_numeric(df["sizeb"], errors="coerce")
    df["org_size"] = sizeb.map(SIZE_LABELS)
    df["org_size_order"] = sizeb.where(sizeb.isin(SIZE_LABELS))

    return df

In [17]:
for year in years:
    data[year] = add_indicators(data[year])

# Stack all waves into one dataframe
all_waves = pd.concat(data.values(), ignore_index=True)

print("Stacked shape:", all_waves.shape)
print("governance_A range:", all_waves["governance_A"].min(), "–", all_waves["governance_A"].max())
print("governance_B range:", all_waves["governance_B"].min(), "–", all_waves["governance_B"].max())
print("Breach flag distribution:")
print(all_waves["breached"].value_counts(dropna=False))

Stacked shape: (21769, 1026)
governance_A range: 0.0 – 1.0
governance_B range: -110.33333333333333 – 1.0
Breach flag distribution:
breached
0.0    11053
1.0    10716
Name: count, dtype: int64


---
### 3. Weighted aggregation helpers

In [18]:
def wpct(series, weights):
    """Weighted proportion of 1s in a 0/1 series (NaN excluded)."""
    mask = series.isin([0.0, 1.0])
    s, w = series[mask], weights[mask]
    return float((s * w).sum() / w.sum()) if w.sum() > 0 else np.nan

def wmean(series, weights):
    """Weighted mean of a numeric series (NaN excluded)."""
    mask = series.notna()
    s, w = series[mask], weights[mask]
    return float((s * w).sum() / w.sum()) if w.sum() > 0 else np.nan

---
### 4. Story 1 — UK posture over 8 years

**Scope:** all org types. **Output:** `year, governance_score_A, governance_score_B, breach_pct, participant_count`

In [19]:
rows = []
for year, df in data.items():
    w = df["weight"]
    rows.append({
        "year": year,
        "governance_score_A": wmean(df["governance_A"], w),
        "governance_score_B": wmean(df["governance_B"], w),
        "breach_pct":         wpct(df["breached"], w),
        "participant_count":  len(df),
    })

story1 = pd.DataFrame(rows).sort_values("year")
print(story1.to_string(index=False))

 year  governance_score_A  governance_score_B  breach_pct  participant_count
 2018            0.283616            0.374039    0.373231               2088
 2019            0.342784            0.437902    0.304159               2080
 2020            0.408494            0.491153    0.452978               1900
 2021            0.383749            0.467158    0.398887               2284
 2022            0.434039            0.515755    0.431726               2157
 2023            0.376050            0.459663    0.354364               3991
 2024            0.399591            0.478609    0.479829               3434
 2025            0.428775           -0.841293    0.434177               3835


---
### 5. Story 2 — Board priority by org size, 2018–2025

**Scope:** businesses only (`samptype == 1`), `sizeb` in {1,2,3,4}.

**Output:** `year, org_size, org_size_order, board_priority_pct` (% with priority 'Very high' or 'Fairly high').
Also includes `board_priority_mean_norm` (weighted mean of normalised priority, 0=lowest, 1=highest) as the continuous alternative.

In [20]:
rows = []
for year, df in data.items():
    biz = df[(df["samptype"] == 1) & df["org_size"].notna()].copy()
    for size_code, size_label in SIZE_LABELS.items():
        grp = biz[biz["org_size_order"] == size_code]
        if len(grp) == 0:
            continue
        w = grp["weight"]
        rows.append({
            "year":                   year,
            "org_size":               size_label,
            "org_size_order":         size_code,
            "board_priority_pct":     wpct(grp["priority_bin"], w),
            "board_priority_mean_norm": wmean(grp["priority_norm"], w),
        })

story2 = pd.DataFrame(rows).sort_values(["year", "org_size_order"])
print(story2.to_string(index=False))

 year org_size  org_size_order  board_priority_pct  board_priority_mean_norm
 2018    Micro               1            0.721905                  0.648841
 2018    Small               2            0.802079                  0.718573
 2018   Medium               3            0.912921                  0.781987
 2018    Large               4            0.920197                  0.828851
 2019    Micro               1            0.778464                  0.699982
 2019    Small               2            0.853611                  0.759432
 2019   Medium               3            0.954194                  0.831178
 2019    Large               4            0.969028                  0.920833
 2020    Micro               1            0.797412                  0.704976
 2020    Small               2            0.879887                  0.785539
 2020   Medium               3            0.941988                  0.846899
 2020    Large               4            0.980383                  0.917443

---
### 6. Story 3 — Pathway analysis: policy → management → breach

**Scope:** all org types, all years pooled. Routing missings excluded.

Pathway nodes:
- `has_policy`: `manage3 == 1` (formal cyber security policy in place)
- `actively_managed`: `manage1 == 1` (board/senior manager with cyber responsibility)
- `breached`: `type11 == 0`

**Output:** 8-row table (all combinations of the three binary flags) with `weighted_n` and `proportion` (of total weighted population).

In [21]:
pathway_cols = ["has_policy", "actively_managed", "breached"]

# Use all-waves stack; drop respondents where any pathway var is NaN
pw = all_waves[pathway_cols + ["weight"]].dropna(subset=pathway_cols).copy()
pw[pathway_cols] = pw[pathway_cols].astype(int)

total_w = pw["weight"].sum()

rows = []
for hp in [0, 1]:
    for am in [0, 1]:
        for br in [0, 1]:
            mask = (pw["has_policy"] == hp) & (pw["actively_managed"] == am) & (pw["breached"] == br)
            wn = pw.loc[mask, "weight"].sum()
            rows.append({
                "has_policy":       hp,
                "actively_managed": am,
                "breached":         br,
                "weighted_n":       wn,
                "proportion":       wn / total_w,
            })

story3 = pd.DataFrame(rows)
print(story3.to_string(index=False))
print("\nProportions sum to:", story3["proportion"].sum())

 has_policy  actively_managed  breached  weighted_n  proportion
          0                 0         0 6998.430744    0.334264
          0                 0         1 3088.306555    0.147506
          0                 1         0 1558.602314    0.074443
          0                 1         1 1215.828217    0.058071
          1                 0         0 1612.869733    0.077035
          1                 0         1 1574.234964    0.075190
          1                 1         0 2221.237731    0.106092
          1                 1         1 2667.351888    0.127400

Proportions sum to: 1.0


---
### 7. Story 4 — Control adoption over time

**Scope:** all org types.

**Output:** `year, ce_adoption_pct, ten_steps_pct, breach_nonadopter_pct`
- `ce_adoption_pct`: weighted % with `allessentials == 1`
- `ten_steps_pct`: weighted % with `any10steps == 1`
- `breach_nonadopter_pct`: breach rate among orgs where both `allessentials == 0` AND `any10steps == 0`

In [22]:
rows = []
for year, df in data.items():
    w = df["weight"]

    ce = pd.to_numeric(df["allessentials"], errors="coerce")
    ce = ce.where(~ce.isin(MISSING_CODES))

    steps = pd.to_numeric(df["any10steps"], errors="coerce")
    steps = steps.where(~steps.isin(MISSING_CODES))

    # Non-adopters: both CE and 10 Steps not adopted
    nonadopter_mask = (ce == 0) & (steps == 0)
    nonadopter_df = df[nonadopter_mask]

    rows.append({
        "year":                 year,
        "ce_adoption_pct":      wpct(ce.where(ce.isin([0, 1])), w),
        "ten_steps_pct":        wpct(steps.where(steps.isin([0, 1])), w),
        "breach_nonadopter_pct": wpct(nonadopter_df["breached"], nonadopter_df["weight"]),
    })

story4 = pd.DataFrame(rows).sort_values("year")
print(story4.to_string(index=False))

 year  ce_adoption_pct  ten_steps_pct  breach_nonadopter_pct
 2018         0.450763       0.944574               0.047487
 2019         0.519816       0.958284               0.068217
 2020         0.531055       0.964865               0.143275
 2021         0.312261       0.923780               0.110392
 2022         0.299748       0.940095               0.115759
 2023         0.224530       0.887050               0.135521
 2024         0.245904       0.911101               0.145826
 2025         0.234372       0.888500               0.115267


---
### 8. Story 5 — Governance score vs breach, by org size

**Scope:** businesses only (`samptype == 1`), `sizeb` in {1,2,3,4}.

**Output:** `year, org_size, org_size_order, governance_score_A, governance_score_B, breach_pct, org_count`

In [23]:
rows = []
for year, df in data.items():
    biz = df[(df["samptype"] == 1) & df["org_size"].notna()].copy()
    for size_code, size_label in SIZE_LABELS.items():
        grp = biz[biz["org_size_order"] == size_code]
        if len(grp) == 0:
            continue
        w = grp["weight"]
        rows.append({
            "year":             year,
            "org_size":         size_label,
            "org_size_order":   size_code,
            "governance_score_A": wmean(grp["governance_A"], w),
            "governance_score_B": wmean(grp["governance_B"], w),
            "breach_pct":       wpct(grp["breached"], w),
            "org_count":        len(grp),
        })

story5 = pd.DataFrame(rows).sort_values(["year", "org_size_order"])
print(story5.to_string(index=False))

 year org_size  org_size_order  governance_score_A  governance_score_B  breach_pct  org_count
 2018    Micro               1            0.254044            0.357578    0.402701        655
 2018    Small               2            0.395210            0.479276    0.474740        349
 2018   Medium               3            0.536821            0.597136    0.644252        263
 2018    Large               4            0.642732            0.688185    0.757893        252
 2019    Micro               1            0.297073            0.402155    0.277647        770
 2019    Small               2            0.436270            0.519696    0.413423        330
 2019   Medium               3            0.610498            0.661785    0.441475        301
 2019    Large               4            0.714690            0.765262    0.464056        214
 2020    Micro               1            0.324323            0.421087    0.435693        644
 2020    Small               2            0.513359          

---
### 9. Save all outputs

In [24]:
outputs = {
    "story1_uk_posture_trends.csv":         story1,
    "story2_board_priority_by_size.csv":    story2,
    "story3_pathway_analysis.csv":          story3,
    "story4_control_adoption.csv":          story4,
    "story5_governance_vs_breach_size.csv": story5,
}

for filename, df in outputs.items():
    path = OUT_TABLES / filename
    df.to_csv(path, index=False)
    print(f"Saved {filename} ({len(df)} rows)")

Saved story1_uk_posture_trends.csv (8 rows)
Saved story2_board_priority_by_size.csv (32 rows)
Saved story3_pathway_analysis.csv (8 rows)
Saved story4_control_adoption.csv (8 rows)
Saved story5_governance_vs_breach_size.csv (32 rows)
